# Narrative Trails sobre bpoil

Línea base del proyecto: el método original de Narrative Trails sobre el subset
bpoil completo (1133 documentos, abril de 2010 a enero de 2011), sin el
truncamiento que se usa con RollingLDA. Hace lo mismo que
`scripts/baseline_narrative.py`, pero deja el landscape en memoria para probar
distintos pares de documentos sin reajustarlo.

El método tiene tres pasos:

1. UMAP proyecta los embeddings (mpnet, 768 dimensiones) a 48 dimensiones y
   HDBSCAN agrupa esa proyección en tópicos. Cada documento queda con una
   distribución de pertenencia a los tópicos.
2. La coherencia entre dos documentos es la media geométrica entre la similitud
   angular de sus embeddings y la similitud de sus distribuciones de tópicos
   (Jensen-Shannon). Se descartan las aristas más débiles que la arista más
   débil del árbol de expansión máxima (la coherencia crítica), lo que deja el
   grafo conexo. Con la restricción de fechas, cada documento solo apunta a
   documentos del mismo día o posteriores.
3. Una narrativa entre un origen y un destino es el camino de capacidad máxima:
   el que maximiza la coherencia de su eslabón más débil (el bottleneck).
   `reliability` es la media geométrica de las coherencias del camino.

Paper y repositorio en las [referencias del README](../README.md#narrative-trails).

In [1]:
import warnings

# Dos avisos que no afectan el resultado: tqdm pide ipywidgets al importarse
# desde umap, y hdbscan 0.8.40 llama a scikit-learn 1.6 con un argumento que
# se renombró ("force_all_finite").
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message=".*force_all_finite.*")

import pandas as pd

from causal_coherence.data_loading import documents_between, load_bpoil_full
from causal_coherence.narrative import (
    alternatives_summary,
    count_topics,
    extract_alternatives,
    load_or_build_landscape,
    node_degree_report,
    print_alternatives,
    random_pair_paths,
)

pd.set_option("display.max_colwidth", None)

REFERENCE_HASH = "63d0faab065a6f4c"  # results/corrida_1.txt y corrida_2.txt

df, embeddings = load_bpoil_full()
landscape, coherence_hash = load_or_build_landscape(embeddings, df["date"])

print(f"{len(df)} documentos, de {df['date'].min().date()} a {df['date'].max().date()}")
print(f"tópicos (HDBSCAN): {count_topics(landscape.cluster_labels)}")
print(f"hash de la matriz de coherencia: {coherence_hash} (igual a la referencia: {coherence_hash == REFERENCE_HASH})")

Landscape cargado desde caché: landscape_c6fc79eb47995ef7.pkl
1133 documentos, de 2010-04-01 a 2011-01-30
tópicos (HDBSCAN): 54
hash de la matriz de coherencia: 63d0faab065a6f4c (igual a la referencia: True)


## Caché del landscape

UMAP, HDBSCAN y el grafo de coherencia tardan unos 20 segundos en este corpus.
`load_or_build_landscape` guarda el landscape ajustado en `data/cache/` (git lo
ignora) y en las corridas siguientes lo carga en menos de un segundo. El nombre
del archivo es un hash de todo lo que determina el resultado: embeddings,
fechas, `LANDSCAPE_PARAMS` y las versiones de las librerías que lo cambian (ver
`docs/reproducibilidad.md`). Si cambia cualquiera de esas cosas, cambia el hash
y el landscape se vuelve a ajustar.

Junto al landscape se guarda el hash de la matriz de coherencia, el mismo
identificador con que `scripts/baseline_narrative.py` nombra sus resultados.

## Los títulos de bpoil

En este corpus ningún documento trae titular. El campo `title` son las primeras
palabras del texto con "…" al final, y el `metadata` de cada documento lo dice
("title derived from lead"). Por eso, en cada narrativa se imprime el título
completo tal como viene y debajo los primeros 240 caracteres del contenido.

In [2]:
derived = df["metadata"].astype(str).str.contains("title derived from lead")
print(f"documentos con título derivado del texto: {derived.sum()} de {len(df)}")

documentos con título derivado del texto: 1133 de 1133


## Dos pares de referencia

El proyecto tiene corridas de referencia para dos pares de extremos. Los dos se
muestran acá porque cada uno ilustra algo distinto; ninguno corrige al otro.

3 → 975 fue el primer par, elegido por su lectura narrativa: el documento 3
(22 de abril de 2010) reporta el hundimiento de la plataforma y el riesgo de
derrame, y el 975 (16 de septiembre de 2010) anuncia el sellado del pozo. Es
el par de `results/corrida_1.txt` y `corrida_2.txt`.

15 → 572 se eligió después, de forma ad hoc, para probar si otro par daba
resultados distintos. Es el par de referencia actual de
`scripts/baseline_narrative.py`, fijado después de revisar el grado de sus
extremos (sección siguiente).

En los dos casos las alternativas se extraen como en el script: después de cada
narrativa se ocultan sus nodos intermedios y se vuelve a buscar, hasta tener
`N_PATHS` caminos distintos o quedarse sin caminos.

In [3]:
N_PATHS = 3

storylines = extract_alternatives(landscape, 3, 975, N_PATHS)
display(alternatives_summary(storylines).round(3))
print_alternatives(df, landscape, storylines, lead_chars=160)

,largo,bottleneck,reliability
alternativa,,,
0,4,0.523,0.716
1,3,0.515,0.644
2,3,0.503,0.636


--- Alternativa 0: 4 documentos · bottleneck 0.523 · reliability 0.716
[3] 2010-04-22 · tópico 41
    Rig sinks in Gulf of Mexico, oil spill risk looms Fire boat…
    > Rig sinks in Gulf of Mexico, oil spill risk looms Fire boat response crews battle the blazing remnants of the off shore oil rig Deepwater Horizon, off Louisiana…
[49] 2010-05-01 · tópico 41 · coherencia 0.890
    Federal and state officials pushed oil giant BP to intensify its efforts…
    > Federal and state officials pushed oil giant BP to intensify its efforts to cap a leaking oil well in the Gulf of Mexico and to contain the slick that is threat…
[324] 2010-06-01 · tópico 41 · coherencia 0.523
    Tue Jun 1, 2010 1:33 pm EDT ( Reuters ) - Millions…
    > Tue Jun 1, 2010 1:33 pm EDT ( Reuters ) - Millions of gallons ( liters ) of oil have poured into the Gulf of Mexico since an April 20 blast on the Deepwater Hor…
[975] 2010-09-16 · tópico 49 · coherencia 0.788
    BP's Gulf of Mexico oil well to be sealed ` by Sunda

Con 3 → 975, las tres alternativas son cortas (3 y 4 documentos) y su bottleneck
ronda 0,5. Los caminos saltan casi directo de fines de abril a septiembre. El
eslabón más débil siempre está cerca del origen (49 → 324 en la primera, la
arista que sale de 3 en las otras dos); la llegada a 975 tiene coherencias de
0,79 a 0,81.

La alternativa 2 pasa por el documento 8, cuyo título es solo "Apr." porque el
texto empieza con "Apr. 21: …". Es un ejemplo del problema de títulos descrito
arriba.

In [4]:
storylines = extract_alternatives(landscape, 15, 572, N_PATHS)
display(alternatives_summary(storylines).round(3))
print_alternatives(df, landscape, storylines, lead_chars=160)

,largo,bottleneck,reliability
alternativa,,,
0,7,0.828,0.849
1,8,0.816,0.848
2,6,0.809,0.827


--- Alternativa 0: 7 documentos · bottleneck 0.828 · reliability 0.849
[15] 2010-04-27 · tópico 39
    LONDON | Tue Apr 27, 2010 4:08 am EDT LONDON ( Reuters…
    > LONDON | Tue Apr 27, 2010 4:08 am EDT LONDON ( Reuters ) - BP Plc ( BP. L ) failed to reassure investors with a more than doubling of first-quarter net profits…
[28] 2010-04-29 · tópico 39 · coherencia 0.876
    LONDON | Thu Apr 29, 2010 11:29 am EDT LONDON ( Reuters…
    > LONDON | Thu Apr 29, 2010 11:29 am EDT LONDON ( Reuters ) - Shares in London-based BP Plc fell 7 percent on Thursday after the oil major said a leaking well in…
[332] 2010-06-02 · tópico 39 · coherencia 0.828
    US attorney general, Eric Holder, confirmed that a criminal and civil investigation…
    > US attorney general, Eric Holder, confirmed that a criminal and civil investigation had been opened Workers in Louisiana tackle oil from the Deepwater Horizon l…
[368] 2010-06-04 · tópico 39 · coherencia 0.834
    LONDON -- As BP shares take a pounding and

Con 15 → 572, los caminos son más largos (6 a 8 documentos) y su bottleneck es
más alto (0,81 a 0,83). Los extremos están separados por 54 días, contra 147 en
3 → 975. Los tres caminos empiezan con varios documentos del tópico 39
(acciones, dividendos y demandas contra BP) y terminan en la crítica a Tony
Hayward. La diferencia entre los dos pares es grande; la sección siguiente
muestra de dónde sale.

## Grado de los extremos

`node_degree_report` compara el grado de un nodo (aristas entrantes más
salientes) con el resto del grafo. `percentile` es la fracción de nodos con
grado estrictamente menor, y `peripheral` marca los nodos con menos de la mitad
del grado promedio. Un extremo con pocas aristas deja al camino pocas opciones
para salir o llegar, y eso se nota en el bottleneck.

In [5]:
PAIR_NODES = [3, 975, 15, 572]

all_degrees = pd.Series(dict(landscape.nx_graph.degree()))
print(f"grado en el grafo: mediana {all_degrees.median():.0f}, promedio {all_degrees.mean():.0f}, "
      f"p10 {all_degrees.quantile(0.1):.0f}, p90 {all_degrees.quantile(0.9):.0f}, máximo {all_degrees.max()}")
print(f"nodos periféricos: {(all_degrees < all_degrees.mean() / 2).sum()} de {len(all_degrees)}")
print(f"nodos con grado entre 860 y el máximo: {all_degrees.between(860, all_degrees.max()).mean():.0%}")

degrees = pd.DataFrame([node_degree_report(landscape, n) for n in PAIR_NODES]).set_index("node")
degrees.insert(0, "fecha", df.loc[PAIR_NODES, "date"].dt.date.values)
degrees.round({"mean_degree": 1, "percentile": 3})

grado en el grafo: mediana 876, promedio 678, p10 15, p90 886, máximo 901
nodos periféricos: 264 de 1133
nodos con grado entre 860 y el máximo: 76%


,fecha,degree,mean_degree,median_degree,percentile,peripheral
node,,,,,,
3,2010-04-22,18,678.4,876.0,0.125,True
975,2010-09-16,875,678.4,876.0,0.434,False
15,2010-04-27,878,678.4,876.0,0.575,False
572,2010-06-20,887,678.4,876.0,0.902,False


Con la restricción de fechas, el grado entrante crece con la fecha (un documento
tardío recibe aristas de casi todos los anteriores) y el saliente baja. Para
saber si un extremo es raro para su momento, la tabla compara sus aristas
entrantes y salientes con la mediana de los documentos a 15 días o menos.

In [6]:
in_degree = pd.Series(dict(landscape.nx_graph.in_degree()))
out_degree = pd.Series(dict(landscape.nx_graph.out_degree()))
rows = []
for node in PAIR_NODES:
    near = df.index[(df["date"] - df.loc[node, "date"]).abs() <= pd.Timedelta(days=15)]
    rows.append({
        "node": node,
        "entrante": in_degree[node],
        "mediana entrante ±15 días": in_degree[near].median(),
        "saliente": out_degree[node],
        "mediana saliente ±15 días": out_degree[near].median(),
    })
pd.DataFrame(rows).set_index("node")

,entrante,mediana entrante ±15 días,saliente,mediana saliente ±15 días
node,,,,
3,0,40.0,18,826.0
975,764,754.0,111,105.0
15,12,60.0,866,807.0
572,470,372.0,417,422.0


Las 18 aristas del documento 3, ordenadas por peso, con el grado de cada vecino:

In [7]:
neighbors = pd.DataFrame([
    {"vecino": t, "peso": d["weight"], "fecha": df.loc[t, "date"].date(),
     "grado": landscape.nx_graph.degree(t),
     "periférico": node_degree_report(landscape, t)["peripheral"],
     "título": df.loc[t, "title"]}
    for t, d in landscape.nx_graph[3].items()
]).sort_values("peso", ascending=False).set_index("vecino")
neighbors.round(3)

,peso,fecha,grado,periférico,título
vecino,,,,,
17,0.898,2010-04-28,25,True,April 27: Weathered oil from a leaking pipeline that resulted the explosion…
807,0.894,2010-07-23,24,True,"July 23 | Fri Jul 23, 2010 7:55 am EDT July 23…"
940,0.891,2010-09-03,20,True,Boats are seen spraying water on an oil and gas platform that…
941,0.891,2010-09-03,20,True,Gulf of Mexico Platform Explosion An oil platform exploded and caught fire…
634,0.891,2010-06-28,25,True,"Mon Jun 28, 2010 12:19 pm EDT ( Reuters ) - Millions…"
49,0.890,2010-05-01,22,True,Federal and state officials pushed oil giant BP to intensify its efforts…
939,0.882,2010-09-02,20,True,"Thursday, September 2, 2010; 11:48 PM An oil and gas production platform…"
775,0.850,2010-07-19,18,True,As many as 60 vessels work the Deepwater Horizon drilling area to…
9,0.515,2010-04-26,880,False,"The slick has now grown to about 1,500 sq km There are…"


El nodo atípico de 3 → 975 es el origen. El documento 3 tiene grado 18 y es
periférico: sus 18 aristas son salientes, cuando los documentos a 15 días o
menos tienen una mediana de 826 aristas salientes. Sus 8 aristas fuertes (0,85
a 0,90) van a documentos que también son periféricos, con grado entre 18 y 25;
por los títulos, varios parecen pies de foto o notas breves, como el propio
documento 3. Hacia el resto del grafo solo tiene aristas de 0,46 a 0,52.
Cualquier camino que salga de 3 tiene que cruzar hacia el grafo principal por
una arista de ese rango, y eso fija el bottleneck cerca de 0,5 sea cual sea el
destino.

El destino 975 es típico: su grado total está en la mediana del grafo, y su
grado entrante alto (764) es el esperable para un documento de septiembre (la
mediana de sus vecinos en el tiempo es 754).

15 y 572 no son periféricos y su grado queda a menos de 2 % de la mediana del
grafo. El percentil 0,90 de 572 parece alto, pero en bpoil el grado está muy
concentrado arriba: el 76 % de los nodos tiene entre 860 y 901 aristas, y ese
percentil equivale a 11 aristas sobre la mediana. En este grafo no hay hubs; el
riesgo está en la cola de nodos con pocas aristas.

Una versión anterior de esta historia decía que el problema era que 975 es un
hub, en el percentil ~81 de grado. Ese número existe, pero es el del documento
975 del corpus de Taliban (una nota de mayo de 2009 sobre las elecciones
afganas). Salía de un print de depuración de `afg_explore.py` que usaba un
índice de bpoil sobre el grafo de Afganistán. En el grafo de bpoil, 975 está en
el percentil 43.

La lección se mantiene: un par elegido con buen criterio narrativo puede tener
un extremo estructuralmente atípico que no se nota leyendo los titulares. Por
eso `scripts/baseline_narrative.py` revisa el grado de los dos extremos en cada
corrida, siempre sobre el grafo del corpus que se está usando.

## Pares al azar en bpoil

El mismo experimento del notebook de Taliban, sobre bpoil: la narrativa entre 20
pares de documentos elegidos al azar (semilla 0), con la separación en días, el
largo y el bottleneck de cada camino, y el percentil de grado de cada extremo.
Es el ciclo que estaba en `scripts/baseline_narrative.py` antes de limpiar los
prints de depuración; con la misma semilla da los mismos pares.

In [8]:
pairs = random_pair_paths(landscape, df["date"], n_pairs=20, seed=0)
pairs["periférico"] = [
    node_degree_report(landscape, s)["peripheral"] or node_degree_report(landscape, t)["peripheral"]
    for s, t in zip(pairs["src"], pairs["tgt"])
]
found = pairs.dropna(subset=["largo"])
print(f"pares con camino: {len(found)} de {len(pairs)}")
print(f"largo: mediana {found['largo'].median():.0f}, media {found['largo'].mean():.1f}, "
      f"mínimo {found['largo'].min():.0f}, máximo {found['largo'].max():.0f}")
print("\nbottleneck según si algún extremo es periférico (NaN = sin camino):")
print(pairs.groupby("periférico")["bottleneck"].agg(["size", "count", "min", "median", "max"]).round(3).to_string())
pairs.round(3)

pares con camino: 19 de 20
largo: mediana 6, media 7.0, mínimo 3, máximo 12

bottleneck según si algún extremo es periférico (NaN = sin camino):
            size  count    min  median    max
periférico                                   
False         15     15  0.733   0.778  0.883
True           5      4  0.481   0.626  0.715


,src,tgt,días,largo,bottleneck,pct_src,pct_tgt,periférico
0,721,962,58,8.0,0.766,0.751,0.312,False
1,305,348,3,5.0,0.774,0.488,0.575,False
2,18,85,7,5.0,0.780,0.395,0.659,False
3,735,920,36,7.0,0.715,0.232,0.395,True
4,570,687,19,6.0,0.749,0.575,0.615,False
5,716,825,13,12.0,0.733,0.857,0.659,False
6,633,1059,134,6.0,0.641,0.575,0.192,True
7,760,923,36,9.0,0.778,0.826,0.434,False
8,446,971,96,9.0,0.770,0.751,0.488,False
9,38,866,94,6.0,0.854,0.488,0.252,False


19 de los 20 pares tienen camino, con un largo típico de 6 a 7 documentos
(mediana 6, media 7,0, entre 3 y 12). Los cinco pares con algún extremo
periférico concentran los peores resultados: uno no tiene camino (595 → 759,
los dos extremos periféricos) y los otros cuatro tienen los cuatro bottleneck
más bajos de la muestra (0,481, 0,611, 0,641 y 0,715). Los 15 pares sin
extremos periféricos tienen bottleneck entre 0,733 y 0,883. Es el mismo patrón
de 3 → 975, visto en pares que nadie eligió.

## Probar otro par

Para probar otros extremos basta con cambiar `SRC_NODE` y `TGT_NODE` y correr
solo la celda: usa el landscape ya cargado. Por la restricción de fechas, el
origen tiene que ser del mismo día o anterior al destino. Las líneas comentadas
son pares que ya se usaron en la investigación, listos para descomentar.
`documents_between` lista candidatos en una ventana de fechas; los índices son
los de `df`, que está ordenado por fecha.

In [9]:
documents_between(df, "2010-04-26", "2010-04-28")

,date,title
9,2010-04-26,"The slick has now grown to about 1,500 sq km There are…"
10,2010-04-26,Robot vessels used to cap Gulf of Mexico oil leak The US…
11,2010-04-26,Debris and oil from the Deepwater Horizon drilling platform float in the…
12,2010-04-26,Industry officials acknowledge it could take months to entirely contain leak from…
13,2010-04-27,April 26: Weathered oil is seen on the surface of the Gulf…
14,2010-04-27,Energy firm beats expectations with # 3.6 bn quarterly profit but performance…
15,2010-04-27,"LONDON | Tue Apr 27, 2010 4:08 am EDT LONDON ( Reuters…"
16,2010-04-27,Gulf of Mexico oil spill creates environmental and political dilemmas View how…
17,2010-04-28,April 27: Weathered oil from a leaking pipeline that resulted the explosion…
18,2010-04-28,ON THE GULF OF MEXICO -- The deadly blowout of an oil…


In [10]:
# Pares ya usados en la investigación; descomentar uno para probarlo.
# SRC_NODE, TGT_NODE = 3, 975     # primer par: origen periférico (grado 18), caminos cortos y débiles
# SRC_NODE, TGT_NODE = 15, 572    # par de referencia de scripts/baseline_narrative.py
# SRC_NODE, TGT_NODE = 716, 825   # muestra al azar: el camino más largo (12 documentos)
# SRC_NODE, TGT_NODE = 595, 759   # muestra al azar: sin camino, los dos extremos periféricos
SRC_NODE, TGT_NODE = 291, 697     # muestra al azar: destino periférico (grado 7), bottleneck 0,48

N_PATHS = 1

assert df.loc[SRC_NODE, "date"] <= df.loc[TGT_NODE, "date"], "el origen tiene que ser anterior al destino"
print({role: round(node_degree_report(landscape, n)["percentile"], 2) for role, n in (("origen", SRC_NODE), ("destino", TGT_NODE))})
storylines = extract_alternatives(landscape, SRC_NODE, TGT_NODE, N_PATHS)
if storylines:
    display(alternatives_summary(storylines).round(3))
    print_alternatives(df, landscape, storylines, lead_chars=160)
else:
    print("No hay camino entre esos dos documentos.")

{'origen': 0.62, 'destino': 0.03}


,largo,bottleneck,reliability
alternativa,,,
0,11,0.481,0.8


--- Alternativa 0: 11 documentos · bottleneck 0.481 · reliability 0.800
[291] 2010-05-30 · tópico 47
    BOOTHVILLE, La.
    > BOOTHVILLE, La. -- BOOTHVILLE, La. ( AP ) -- The reality that the Gulf oil leak could keep flowing for months was setting in for some somber churchgoers in Loui…
[308] 2010-05-31 · tópico 47 · coherencia 0.832
    Gulf oil spill threat widens, BP shares drop VENICE, La.
    > Gulf oil spill threat widens, BP shares drop VENICE, La. | Mon May 31, 2010 11:12 pm IST VENICE, La. ( Reuters ) - Oil from BP's out-of-control Gulf of Mexico o…
[371] 2010-06-05 · tópico 45 · coherencia 0.846
    VENICE, La\/PENSACOLA BEACH, Fla | Sat Jun 5, 2010 7:16 pm EDT…
    > VENICE, La\/PENSACOLA BEACH, Fla | Sat Jun 5, 2010 7:16 pm EDT VENICE, La\/PENSACOLA BEACH, Fla ( Reuters ) - The latest effort to siphon oil and gas gushing fr…
[373] 2010-06-06 · tópico 53 · coherencia 0.832
    BP cap captures ' 10,000 barrels ' a day in US Gulf…
    > BP cap captures ' 10,000 barrels ' a da